In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import nibabel as nib
import numpy as np
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from TransAttUnet import UNet_Attention_Transformer_Multiscale

zipファイルの解凍

In [15]:
import zipfile
import os

# アップロードしたzipファイル名に合わせて書き換えてください
zip_file_path = '/workspace/NSCLC_NIfTI2/imagesTr/LUNG1-301-422.nii.zip' 
# 解凍先のフォルダ
extract_dir = '/workspace/NSCLC_NIfTI2/imagesTr'

if os.path.exists(zip_file_path):
    print(f"{zip_file_path} を解凍中...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"解凍が完了しました！場所: {extract_dir}")
else:
    print(f"エラー: {zip_file_path} が見つかりません。アップロードを確認してください。")

/workspace/NSCLC_NIfTI2/imagesTr/LUNG1-301-422.nii.zip を解凍中...
解凍が完了しました！場所: /workspace/NSCLC_NIfTI2/imagesTr


In [4]:
import os
print(os.getcwd())        # 現在の場所を表示
print(os.listdir('.'))

/workspace/TransAttUnet/model
['__pycache__', 'unet_parts_att_transformer.py', 'TransAttUnet.py', 'train_nsclc.py', 'nsclc.ipynb', 'unet_parts_att_multiscale.py', 'unet_parts.py']


In [30]:
class NSCLCNiftiDataset(Dataset):
    def __init__(self, root_dir, file_list, target_size=(512, 512)):
        self.img_dir = os.path.join(root_dir, 'imagesTr')
        self.mask_dir = os.path.join(root_dir, 'labelsTr')
        self.file_names = file_list
        self.target_size = target_size
    
    def __len__(self):
        return len(self.file_names)
    
    def __getitem__(self, idx):
        # NIfTI読み込み
        img_path = os.path.join(self.img_dir, self.file_names[idx])
        mask_path = os.path.join(self.mask_dir, self.file_names[idx])

        img_nii = nib.load(img_path).get_fdata()
        mask_nii = nib.load(mask_path).get_fdata()

        # 3Dから2Dスライス(腫瘍がいちばん大きいスライスとる)
        # 配列の中で最大値となってる要素のうち先頭のインデックスを返す(argmax)
        mask_slice_sums = np.argmax(np.sum(mask_nii, axis=(0, 1)))

        max_mask_idx = np.argmax(mask_slice_sums)

        # 画像とlabelのshapeが合わないから、画像とラベルの枚数比の計算
        z_ratio = img_nii.shape[2] / mask_nii.shape[2]
        # 対応するインデックスを割り出す
        max_img_idx = int(max_mask_idx * z_ratio)
        # 範囲外ガード?
        max_img_idx = min(max_img_idx, img_nii.shape[2] - 1)

        img_slice = img_nii[:, :, max_img_idx]
        mask_slice = mask_nii[:, :, max_img_idx]

        # CT値の正規化(?わからん -1000~400程度にクリップ)
        # 肺の微細な構造を鮮明に観察するための肺野条件らしい
        img_slice = np.clip(img_slice, -1000, 400)
        #0-1の正規化
        img_slice = (img_slice + 1000) / 1400

        # リサイズとテンソル化
        img_tensor = torch.from_numpy(img_slice).float().unsqueeze(0) # [1, H, W]
        mask_tensor = torch.from_numpy(mask_slice).float().unsqueeze(0)

        img_tensor = F.interpolate(img_tensor.unsqueeze(0), size=self.target_size, mode='bilinear', align_corners=False).squeeze(0)
        mask_tensor = F.interpolate(mask_tensor.unsqueeze(0), size=self.target_size, mode='nearest').squeeze(0)

        return img_tensor, mask_tensor

if __name__ == '__main__':
    root_dir = '/workspace/NSCLC_NIfTI2'

    images_dir = '/workspace/NSCLC_NIfTI2/imagesTr'
    labels_dir = '/workspace/NSCLC_NIfTI2/labelsTr'

    # 全てのファイル名を取得
    all_images = sorted([f for f in os.listdir(images_dir) if f.endswith('.nii.gz')])

    # ラベル側
    valid_files = [f for f in all_images if os.path.exists(os.path.join(labels_dir, f))]

    print(f"画像総数: {len(all_images)}")
    print(f"ラベルがある有効な数: {len(valid_files)}")
    print(f"除外されたファイル: {set(all_images) - set(valid_files)}")

    # train 8  test 2で分割
    train_files, test_files = train_test_split(valid_files, test_size=0.2, random_state=42)

    print(f"--- データ分割完了 ---")
    print(f"学習用 (train): {len(train_files)} 症例")
    print(f"テスト用 (test): {len(test_files)} 症例")

画像総数: 422
ラベルがある有効な数: 421
除外されたファイル: {'LUNG1-128.nii.gz'}
--- データ分割完了 ---
学習用 (train): 336 症例
テスト用 (test): 85 症例


In [31]:
# 学習設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = NSCLCNiftiDataset(root_dir=root_dir, file_list=train_files)
test_dataset = NSCLCNiftiDataset(root_dir=root_dir, file_list=test_files)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = UNet_Attention_Transformer_Multiscale(n_channels=1, n_classes=1).to(device)
#最適化アルゴリズム
optimizer = optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-4)
# 損失関数
criterion = nn.BCEWithLogitsLoss()

num_epochs = 50

train_loss_history = []
val_loss_history = []

下のは動かさなくていい(確認用)

In [14]:
import nibabel as nib
import os

images_dir = '/workspace/NSCLC_NIfTI/NSCLC_NIfTI/labelsTr'
all_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.nii.gz')])

broken_files = []
for f in all_files:
    try:
        img = nib.load(os.path.join(images_dir, f))
        _ = img.get_fdata() # 実際にデータを読み込んでチェック
    except Exception as e:
        print(f"壊れたファイルを発見: {f} ({e})")
        broken_files.append(f)

print(f"計 {len(broken_files)} 件の壊れたファイルが見つかりました。")

計 0 件の壊れたファイルが見つかりました。


In [16]:
import os
sample_file = '/workspace/NSCLC_NIfTI2/imagesTr/LUNG1-422.nii.gz'
if os.path.exists(sample_file):
    size = os.path.getsize(sample_file)
    print(f"ファイルサイズ: {size / (1024*1024):.2f} MB")

ファイルサイズ: 28.54 MB


In [26]:
import os
import nibabel as nib
import numpy as np

images_dir = '/workspace/NSCLC_NIfTI2/imagesTr'
labels_dir = '/workspace/NSCLC_NIfTI2/labelsTr'
corrupted_files = []
mismatched_files = []

print("整合性チェックを開始します...")

# imagesTrにあるファイルを基準にチェック
for filename in sorted(os.listdir(images_dir)):
    if not filename.endswith('.nii.gz'):
        continue
        
    img_path = os.path.join(images_dir, filename)
    label_path = os.path.join(labels_dir, filename)
    
    try:
        # 1. ファイルが正常に開けるか（破損チェック）
        img = nib.load(img_path)
        img_data = img.get_fdata()
        
        # ラベルがあるか確認
        if not os.path.exists(label_path):
            print(f"  [欠損] ラベルがありません: {filename}")
            continue
            
        mask = nib.load(label_path)
        mask_data = mask.get_fdata()
        
        # 2. サイズ（スライス数）が一致しているか
        if img_data.shape != mask_data.shape:
            mismatched_files.append((filename, img_data.shape, mask_data.shape))
            
    except Exception as e:
        # 読み込み中にエラーが出たら「破損」とみなす
        print(f"  [破損] 読み込み失敗: {filename} - {str(e)}")
        corrupted_files.append(filename)

print("\n--- チェック完了 ---")
print(f"スキャン総数: {len(os.listdir(images_dir))}")
print(f"読み込み失敗（破損）: {len(corrupted_files)} 件")
print(f"サイズ不一致: {len(mismatched_files)} 件")

# 不一致があった場合、具体的に表示
if mismatched_files:
    print("\n【詳細: サイズ不一致の例】")
    for name, img_shape, mask_shape in mismatched_files[:5]: # 最初の5件だけ表示
        print(f"{name}: 画像{img_shape} vs ラベル{mask_shape}")

整合性チェックを開始します...
  [欠損] ラベルがありません: LUNG1-128.nii.gz

--- チェック完了 ---
スキャン総数: 426
読み込み失敗（破損）: 0 件
サイズ不一致: 416 件

【詳細: サイズ不一致の例】
LUNG1-001.nii.gz: 画像(512, 512, 134) vs ラベル(512, 512, 536)
LUNG1-002.nii.gz: 画像(512, 512, 111) vs ラベル(512, 512, 666)
LUNG1-003.nii.gz: 画像(512, 512, 107) vs ラベル(512, 512, 642)
LUNG1-004.nii.gz: 画像(512, 512, 114) vs ラベル(512, 512, 570)
LUNG1-005.nii.gz: 画像(512, 512, 91) vs ラベル(512, 512, 546)


ここから

In [ ]:
print("学習を開始...")
for epoch in range(num_epochs):
    # 学習
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        # 予測と計算
        outputs = model(images)
        loss = criterion(outputs, masks)

        # 重みの更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss /len(train_loader)

        # 検証(テストデータでの評価)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            v_loss = criterion(outputs, masks)
            val_loss += v_loss.item()

    avg_val_loss = val_loss / len(test_loader)

    train_loss_history.append(avg_train_loss)
    val_loss_history.append(avg_val_loss)

    # 進捗を表示
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # モデルの保存
    os.makedirs('checkpoints', exist_ok=True)

    # 10エポックごとに重みを保存
    if (epoch + 1) % 10 == 0:
        save_path = f"checkpoints/model_epoch_{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"--- モデルを保存しました: {save_path}")

print("すべての学習が完了しました。")

学習を開始...
Epoch [1/50] Train Loss: 0.4856, Val Loss: 0.2800
Epoch [2/50] Train Loss: 0.2009, Val Loss: 0.1499
Epoch [3/50] Train Loss: 0.1207, Val Loss: 0.0995
Epoch [4/50] Train Loss: 0.0853, Val Loss: 0.0738
Epoch [5/50] Train Loss: 0.0653, Val Loss: 0.0573
Epoch [6/50] Train Loss: 0.0530, Val Loss: 0.0477
Epoch [7/50] Train Loss: 0.0442, Val Loss: 0.0405
Epoch [8/50] Train Loss: 0.0381, Val Loss: 0.0351
Epoch [9/50] Train Loss: 0.0332, Val Loss: 0.0315
Epoch [10/50] Train Loss: 0.0295, Val Loss: 0.0274
--- モデルを保存しました: checkpoints/model_epoch_10.pth
Epoch [11/50] Train Loss: 0.0266, Val Loss: 0.0252
Epoch [12/50] Train Loss: 0.0241, Val Loss: 0.0228
Epoch [13/50] Train Loss: 0.0221, Val Loss: 0.0208
